In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import preprocessing
import matplotlib.pyplot as plt
%matplotlib inline

import scipy.stats as stats
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_predict

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import cross_val_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import re


from scipy.spatial import cKDTree

In [2]:
df = pd.read_csv('plant_pest_dataset.csv')
df.shape

(298126, 16)

In [3]:
df.head(30)

,id,observed_on,latitude,longitude,phenophase,scientific_name,common_name,taxon_id,doy,biome,T2M,PRECTOTCORR,SWGDN,biome_cat,scientific_name_pest,common_name_pest
0,310657266,2025-08-31,40.799434,-111.012697,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,292.79025,0.000011,288.58110,H,Empoasca fabae,Potato Leafhopper
1,310812803,2025-08-31,40.690640,-110.903167,Flowering,Achillea millefolium,common yarrow,52821,243,27.0,290.41034,0.000012,279.16245,H,Ostrinia nubilalis,European Corn Borer Moth
2,310648485,2025-08-31,43.939639,-87.719908,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,294.50253,0.000030,255.68756,H,Popillia japonica,Japanese Beetle
3,310647240,2025-08-31,44.741903,-65.519220,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,290.05582,0.000014,265.07320,H,Lymantria dispar,Spongy Moth
4,310482483,2025-08-31,42.146311,-77.131258,Flowering,Achillea millefolium,common yarrow,52821,243,25.0,293.99230,0.000021,266.84604,H,Leptinotarsa decemlineata,Colorado Potato Beetle
5,310457618,2025-08-31,58.544962,31.377577,Flowering,Achillea millefolium,common yarrow,52821,243,26.0,288.90690,0.000045,192.62874,H,Leptinotarsa decemlineata,Colorado Potato Beetle
6,310215347,2025-08-30,63.558598,26.662874,Flowering,Achillea millefolium,common yarrow,52821,242,27.0,287.83690,0.000021,182.90506,H,Plutella xylostella,Diamondback Moth
7,310201801,2025-08-30,60.234195,24.946592,Flowering,Achillea millefolium,common yarrow,52821,242,26.0,289.69420,0.000028,196.11778,H,Cydia pomonella,Codling Moth
8,311993972,2025-08-30,42.558192,2.112421,Flowering,Achillea millefolium,common yarrow,52821,242,26.0,290.25610,0.000025,251.79330,H,Lymantria dispar,Spongy Moth
9,311993941,2025-08-30,42.419870,2.010164,Flowering,Achillea millefolium,common yarrow,52821,242,26.0,290.25610,0.000025,251.79330,H,Lymantria dispar,Spongy Moth


In [4]:
df['observed_on'].nunique()

2542

In [5]:
df['scientific_name_pest'].nunique()

33

In [6]:
pip install pandas numpy scikit-learn scipy plotly geopandas pydeck chord

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- 1. Read Your Dataset ---
print("Reading data from your file...")
try:
    df = pd.read_csv('plant_pest_dataset.csv')
    if 'biome_cat' not in df.columns or 'common_name_pest' not in df.columns:
        raise ValueError("CSV must contain 'biome_cat' and 'common_name_pest' columns.")
    print("Data loaded successfully.")
except FileNotFoundError:
    print("ERROR: File not found. Please make sure 'plant_pest_dataset.csv' is in the correct directory.")
    exit()
except ValueError as e:
    print(f"ERROR: {e}")
    exit()

# --- Filter for Top 10 Pests ---
print("Filtering for the top 10 most frequent pests...")
pest_totals = df['common_name_pest'].value_counts()
top_10_pests = pest_totals.nlargest(10).index.tolist()
df = df[df['common_name_pest'].isin(top_10_pests)]
print("Filtering complete.")

# --- 2. Aggregate Data ---
sighting_counts = df.groupby(['biome_cat', 'common_name_pest']).size().reset_index(name='count')

# --- 3. Create the 3D Mesh for the Bars ---
print("Generating 3D mesh...")
all_biomes = sighting_counts['biome_cat'].unique()
all_pests = sighting_counts['common_name_pest'].unique()

biome_map = {name: i for i, name in enumerate(all_biomes)}
pest_map = {name: i for i, name in enumerate(all_pests)}

sighting_counts['pest_color_val'] = sighting_counts['common_name_pest'].map(pest_map)

all_x, all_y, all_z = [], [], []
all_i, all_j, all_k = [], [], []
vertex_offset = 0
bar_width = 0.4 

for index, row in sighting_counts.iterrows():
    x_pos = biome_map[row['biome_cat']]
    y_pos = pest_map[row['common_name_pest']]
    z_height = row['count']
    
    vertices = [
        [x_pos - bar_width, y_pos - bar_width, 0], [x_pos + bar_width, y_pos - bar_width, 0],
        [x_pos + bar_width, y_pos + bar_width, 0], [x_pos - bar_width, y_pos + bar_width, 0],
        [x_pos - bar_width, y_pos - bar_width, z_height], [x_pos + bar_width, y_pos - bar_width, z_height],
        [x_pos + bar_width, y_pos + bar_width, z_height], [x_pos - bar_width, y_pos + bar_width, z_height]
    ]
    
    for v in vertices:
        all_x.append(v[0])
        all_y.append(v[1])
        all_z.append(v[2])
    
    faces = [
        [0, 1, 2], [0, 2, 3], [4, 5, 1], [4, 1, 0], [7, 6, 5], [7, 5, 4],
        [3, 2, 6], [3, 6, 7], [1, 5, 6], [1, 6, 2], [4, 0, 3], [4, 3, 7]
    ]
    
    for f in faces:
        all_i.append(f[0] + vertex_offset)
        all_j.append(f[1] + vertex_offset)
        all_k.append(f[2] + vertex_offset)

    vertex_offset += 8

# --- NEW: Function to wrap long labels ---
def wrap_labels(labels, max_length=15):
    wrapped_labels = []
    for label in labels:
        if len(label) > max_length:
            # Find the best space to break the line
            space_index = label.rfind(' ', 0, max_length)
            if space_index > 0:
                label = label[:space_index] + '<br>' + label[space_index+1:]
        wrapped_labels.append(label)
    return wrapped_labels

pest_tick_labels = wrap_labels(list(pest_map.keys()))

# --- 4. Create the Figure ---
fig = go.Figure(data=[go.Mesh3d(
    x=all_x, y=all_y, z=all_z,
    i=all_i, j=all_j, k=all_k,
    intensity=np.repeat(sighting_counts['pest_color_val'], 12*3),
    colorscale='Plasma',
    colorbar=dict(
        title='Pest Species',
        tickvals=list(pest_map.values()),
        ticktext=list(pest_map.keys())
    ),
    showscale=True,
    opacity=0.85
)])

# --- 5. Update Layout with Label Fixes ---
fig.update_layout(
    title='3D "Prism" Chart of Top 10 Pest Sightings by Biome',
    # IMPROVEMENT 1: Give the chart more space
    width=1200,
    height=900,
    scene=dict(
        xaxis=dict(
            title='Biome Category',
            tickvals=list(biome_map.values()),
            ticktext=list(biome_map.keys()),
            # IMPROVEMENT 3: Adjust font size
            tickfont=dict(size=10)
        ),
        yaxis=dict(
            title='Pest Name',
            tickvals=list(pest_map.values()),
            # IMPROVEMENT 2: Use wrapped labels
            ticktext=pest_tick_labels,
            # IMPROVEMENT 3: Adjust font size
            tickfont=dict(size=10)
        ),
        zaxis_title='Total Sightings',
        camera=dict(
            eye=dict(x=1.8, y=-1.8, z=1.5)
        )
    ),
    # IMPROVEMENT 1: Adjust margins for labels
    margin=dict(l=0, r=0, b=100, t=50)
)
fig.show()
print("Chart generation complete.")

Reading data from your file...
Data loaded successfully.
Filtering for the top 10 most frequent pests...
Filtering complete.
Generating 3D mesh...


Chart generation complete.


In [8]:
import plotly.io as pio

# (Your existing code to create the 'fig' object)
# ...

# Instead of fig.show(), save it to a JSON file
pio.write_json(fig, 'biome_pest_prism.json')

print("Chart has been saved to my_3d_chart.json")


Chart has been saved to my_3d_chart.json


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px # Imported for dynamic color palettes

def create_marimekko_chart(df, category_col, subcategory_col):
    """
    Generates a dynamic Marimekko chart from a pandas DataFrame

    Args:
        df (pd.DataFrame): The input DataFrame.
        category_col (str): The name of the column for the main categories (e.g., 'biome_cat').
                            These will be the main bars on the y-axis.
        subcategory_col (str): The name of the column for the subcategories to be stacked
                               (e.g., 'common_name_pest').

    Returns:
        go.Figure: A Plotly figure object representing the Marimekko chart.
    """
    print(f"Generating Marimekko chart for '{category_col}' and '{subcategory_col}'...")

    # --- 1. Calculate Proportions and Widths (Dynamically) ---
    # Calculate total counts per category to determine bar widths
    category_totals = df[category_col].value_counts()
    total_records = len(df)
    category_widths = category_totals / total_records

    # Calculate the percentage of each subcategory within each category for segment heights
    proportions = df.groupby([category_col, subcategory_col]).size().unstack(fill_value=0)
    proportions = proportions.apply(lambda x: x / x.sum(), axis=1)

    # --- 2. Construct the Chart (Dynamically) ---
    fig = go.Figure()
    
    # Get a list of all unique subcategories to loop through
    subcategories = proportions.columns
    
    # Generate a dynamic color map using a Plotly color scale
    colors = {subcat: color for subcat, color in zip(subcategories, px.colors.qualitative.Plotly)}

    for subcat in subcategories:
        # The width of each segment is its proportion within the category * the overall width of the category
        segment_widths = proportions[subcat] * category_widths[proportions.index]
        
        # Use f-strings and .title() for clean, dynamic hover labels
        hover_text = (
            f'<b>{category_col.replace("_", " ").title()}:</b> %{{y}}<br>'
            f'<b>{subcategory_col.replace("_", " ").title()}:</b> {subcat}<br>'
            '<b>Proportion:</b> %{x:.1%}<extra></extra>'
        )

        fig.add_trace(go.Bar(
            y=proportions.index,
            x=segment_widths,
            name=subcat,
            orientation='h',
            marker_color=colors.get(subcat, '#CCCCCC'), # Use a default color if not in map
            hovertemplate=hover_text
        ))

    # --- 3. Update Layout (Dynamically) ---
    fig.update_layout(
        barmode='stack',
        title=f'{subcategory_col.replace("_", " ").title()} Distribution by {category_col.replace("_", " ").title()}',
        xaxis=dict(
            title='Proportion of Total Records', 
            tickformat='.0%'
        ),
        yaxis=dict(
            title=f'{category_col.replace("_", " ").title()}'
        ),
        legend_title=f'{subcategory_col.replace("_", " ").title()}'
    )
    
    print("Chart generation complete.")
    return fig

# --- EXAMPLE USAGE ---

# 1. Generate Synthetic Data (or load your own)
print("Generating synthetic data for demonstration...")
num_records = 5000
df = pd.DataFrame({
    'biome_cat': np.random.choice(['Forest', 'Cropland', 'Grassland', 'Wetland'], num_records, p=[0.2, 0.45, 0.3, 0.05]),
    'common_name_pest': np.random.choice(['Corn Earworm', 'Spider Mite', 'Japanese Beetle', 'Aphid', 'Thrip'], num_records, p=[0.4, 0.25, 0.15, 0.1, 0.1]),
})
print("Data generated.")

# 2. Call the function with your DataFrame and column names
#    The function handles everything else internally.
fig = create_marimekko_chart(df, category_col='biome_cat', subcategory_col='common_name_pest')

# 3. Show the figure
fig.show()

Generating synthetic data for demonstration...
Data generated.
Generating Marimekko chart for 'biome_cat' and 'common_name_pest'...
Chart generation complete.


In [10]:
import pandas as pd
import plotly.graph_objects as go

# -------------------------------------------------------------------------- #
#                         DYNAMIC SANKEY DIAGRAM FUNCTION                    #
# -------------------------------------------------------------------------- #

def create_sankey_chart(df, flow_columns):
    """
    Generates a dynamic Sankey diagram from a DataFrame based on a specified flow.

    Args:
        df (pd.DataFrame): The input DataFrame.
        flow_columns (list of str): A list of column names defining the Sankey flow.

    Returns:
        go.Figure: A Plotly figure object for the Sankey diagram.
    """
    print(f"Generating Sankey chart for the flow: {' -> '.join(flow_columns)}...")

    # --- 1. Prepare Data for Sankey (Dynamically) ---
    # Find all unique items in the flow columns to create the nodes
    all_nodes = list(pd.unique(df[flow_columns].values.ravel('K')))
    node_map = {node: i for i, node in enumerate(all_nodes)}

    # Create links for each step in the flow
    all_links_list = []
    for i in range(len(flow_columns) - 1):
        source_col, target_col = flow_columns[i], flow_columns[i+1]
        links = df.groupby([source_col, target_col]).size().reset_index(name='count')
        links['source'] = links[source_col].map(node_map)
        links['target'] = links[target_col].map(node_map)
        all_links_list.append(links)

    all_links = pd.concat(all_links_list, axis=0)

    # --- 2. Create the Sankey Diagram ---
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=all_nodes,
        ),
        link=dict(
            source=all_links['source'],
            target=all_links['target'],
            value=all_links['count']
        )
    )])

    # Create a dynamic title for the chart
    flow_title = ' → '.join([col.replace('_', ' ').title() for col in flow_columns])
    fig.update_layout(title_text=f"Sankey Diagram: {flow_title}", font_size=12)
    
    print("Sankey diagram generation complete.")
    return fig

# -------------------------------------------------------------------------- #
#                              MAIN SCRIPT                                   #
# -------------------------------------------------------------------------- #

# --- Step 1: Load your data ---
df = pd.read_csv('plant_pest_dataset.csv')
print(f"Original dataset size: {len(df)} rows")


# --- Step 2: Filter for Top 10 Plants and Pests ---
# Find the top 10 most common plant names
top_10_plants = df['common_name'].value_counts().nlargest(10).index
print("\nTop 10 most common plants:")
print(list(top_10_plants))

# Find the top 10 most common pest names
top_10_pests = df['common_name_pest'].value_counts().nlargest(10).index
print("\nTop 10 most common pests:")
print(list(top_10_pests))

# Filter the DataFrame to keep only rows with these top plants AND top pests
df_filtered = df[df['common_name'].isin(top_10_plants) & df['common_name_pest'].isin(top_10_pests)].copy()
print(f"\nFiltered dataset size: {len(df_filtered)} rows")
print("-" * 30)


# --- Step 3: Generate and show the Sankey Diagram ---
# Define the flow using column names from your filtered DataFrame
sankey_flow = ['biome_cat', 'common_name', 'common_name_pest']
sankey_fig = create_sankey_chart(df_filtered, flow_columns=sankey_flow)
sankey_fig.show()

Original dataset size: 298126 rows

Top 10 most common plants:
['Oriental bittersweet', 'Jack-in-the-Pulpit', 'partridgeberry', 'wild geranium', 'ribwort plantain', 'purple loosestrife', 'common buckthorn', 'Canadian bunchberry', 'Canada mayflower', 'oxeye daisy']

Top 10 most common pests:
['Emerald Ash Borer', 'Grape Phylloxera', 'Cabbage Aphid', 'Indian-Meal Moth', 'Spotted-winged Drosophila', 'Potato Leafhopper', 'Colorado Potato Beetle', 'Codling Moth', 'Cottony cushion scale', 'European Corn Borer Moth']

Filtered dataset size: 27226 rows
------------------------------
Generating Sankey chart for the flow: biome_cat -> common_name -> common_name_pest...
Sankey diagram generation complete.


In [11]:
output_filename = 'snakey_diagram_biome_common_name_pest.json'
df_filtered.to_json(output_filename, orient='records', indent=4)
print(f"\nFiltered data has been saved to '{output_filename}'")


Filtered data has been saved to 'snakey_diagram_biome_common_name_pest.json'


In [12]:
import pandas as pd
import plotly.express as px
import reverse_geocoder as rg
import pycountry

# --- Step 1: Load your dataset ---
df = pd.read_csv('plant_pest_dataset.csv')
print(f"🌍 Dataset loaded with {len(df)} records.")

# --- Step 2: Convert Coordinates to 2-Letter Country Codes ---
print("🔎 Converting coordinates to country names...")
coords = list(zip(df['latitude'], df['longitude']))
results = rg.search(coords)
df['country_code_2_letter'] = [res['cc'] for res in results]
print("✅ 2-letter country code conversion complete.")

# --- Step 3: Convert 2-Letter Codes to 3-Letter Codes ---
print("🔄 Converting 2-letter codes to 3-letter codes for plotting...")

def get_iso3_code(iso2_code):
    """Converts a 2-letter ISO code to a 3-letter ISO code."""
    try:
        return pycountry.countries.get(alpha_2=iso2_code).alpha_3
    except AttributeError:
        # Return None if the code is not found
        return None

df['country_code_3_letter'] = df['country_code_2_letter'].apply(get_iso3_code)
df.dropna(subset=['country_code_3_letter'], inplace=True)
print("✅ 3-letter code conversion complete.")


# --- Step 4: Prepare Data for Animation ---
top_20_pests = df['common_name_pest'].value_counts().nlargest(20).index
df_top_pests = df[df['common_name_pest'].isin(top_20_pests)]
print(f"\nFiltered dataset to focus on the top 20 pests ({len(df_top_pests)} records).")

pest_country_counts = df_top_pests.groupby(['country_code_3_letter', 'common_name_pest']).size().reset_index(name='sightings')
pest_country_counts.rename(columns={'country_code_3_letter': 'country'}, inplace=True)


# --- Step 5: Create the Animated Choropleth Map ---
print("\n🎨 Generating Animated Choropleth world map...")

fig = px.choropleth(
    pest_country_counts,
    locations="country",
    color="sightings",
    hover_name="country",
    animation_frame="common_name_pest",
    color_continuous_scale=px.colors.sequential.YlOrRd,
    title="Global Sightings by Pest"
)

max_sightings = pest_country_counts['sightings'].max()
fig.update_layout(
    title_x=0.5,
    geo=dict(
        showframe=False,
        showcoastlines=False,
        projection_type='natural earth'
    ),
    coloraxis_colorbar=dict(
        title="Sightings"
    ),
    coloraxis=dict(cmin=0, cmax=max_sightings)
)

fig.show()
print("✅ Map generation complete.")


# --- Step 6: NEW - Save the Figure to Both HTML and JSON ---
# Save as an HTML file (Recommended - this one is the interactive chart)
html_filename = "animated_pest_map.html"
fig.write_html(html_filename)
print(f"\n💾 Interactive animation saved to '{html_filename}'")

# Save as a JSON file (This is the raw data and plot structure)
json_filename = 'animated_map_figure.json'
fig.to_json(json_filename)
print(f"📝 Plot structure saved to '{json_filename}'")

🌍 Dataset loaded with 298126 records.
🔎 Converting coordinates to country names...
Loading formatted geocoded file...
✅ 2-letter country code conversion complete.
🔄 Converting 2-letter codes to 3-letter codes for plotting...
✅ 3-letter code conversion complete.

Filtered dataset to focus on the top 20 pests (263190 records).

🎨 Generating Animated Choropleth world map...


✅ Map generation complete.

💾 Interactive animation saved to 'animated_pest_map.html'
📝 Plot structure saved to 'animated_map_figure.json'


In [13]:
# Save as a JSON file (This is the raw data and plot structure)
json_filename = 'animated_map_figure.json'
fig.to_json(json_filename)
print(f"📝 Plot structure saved to '{json_filename}'")

# Save as a JSON file (This is the raw data and plot structure)
json_filename = 'animated_map_figure.json'
fig.to_json(json_filename)
print(f"📝 Plot structure saved to '{json_filename}'")

📝 Plot structure saved to 'animated_map_figure.json'
📝 Plot structure saved to 'animated_map_figure.json'


In [14]:
import pandas as pd
import plotly.express as px
import reverse_geocoder as rg
import pycountry
import os # Imported the 'os' module to handle file paths

# --- Step 1: Load and Limit Your Dataset ---
df = pd.read_csv('plant_pest_dataset.csv')
print(f"🌍 Full dataset loaded with {len(df)} records.")

# MODIFIED: Limiting the dataframe to the first 1000 observations
df = df.head(1000)
print(f"🔬 Analysis limited to the first {len(df)} records.")


# --- Step 2: Convert Coordinates to Country Codes ---
print("🔎 Converting coordinates to country names...")
coords = list(zip(df['latitude'], df['longitude']))
results = rg.search(coords)
df['country_code_2_letter'] = [res['cc'] for res in results]
print("✅ 2-letter country code conversion complete.")

# --- Step 3: Convert 2-Letter to 3-Letter Codes ---
print("🔄 Converting 2-letter codes to 3-letter codes for plotting...")
def get_iso3_code(iso2_code):
    """Converts a 2-letter ISO code to a 3-letter ISO code."""
    try:
        return pycountry.countries.get(alpha_2=iso2_code).alpha_3
    except AttributeError:
        return None

df['country_code_3_letter'] = df['country_code_2_letter'].apply(get_iso3_code)
df.dropna(subset=['country_code_3_letter'], inplace=True)
print("✅ 3-letter code conversion complete.")


# --- Step 4: Prepare Data for Animation (for Plants) ---
# MODIFIED: Find the top 20 most common plants in the limited dataset
top_20_plants = df['common_name'].value_counts().nlargest(20).index
df_top_plants = df[df['common_name'].isin(top_20_plants)]
print(f"\nFiltered dataset to focus on the top 20 plants ({len(df_top_plants)} records).")

# Count sightings PER PLANT, PER COUNTRY using the 3-letter code
plant_country_counts = df_top_plants.groupby(['country_code_3_letter', 'common_name']).size().reset_index(name='sightings')
plant_country_counts.rename(columns={'country_code_3_letter': 'country'}, inplace=True)


# --- Step 5: Create the Animated Choropleth Map for Plants ---
print("\n🎨 Generating Animated Choropleth world map for Plants...")

fig = px.choropleth(
    plant_country_counts,
    locations="country",
    color="sightings",
    hover_name="country",
    animation_frame="common_name",
    color_continuous_scale=px.colors.sequential.Greens,
    # MODIFIED: Updated title
    title="Global Sightings by Plant (Top 20 from 1000 Observations)"
)

# Adjust layout and set a consistent color scale range
max_sightings = plant_country_counts['sightings'].max()
fig.update_layout(
    title_x=0.5,
    geo=dict(
        showframe=False,
        showcoastlines=False,
        projection_type='natural earth'
    ),
    coloraxis_colorbar=dict(
        title="Sightings"
    ),
    coloraxis=dict(cmin=0, cmax=max_sightings)
)

# --- Step 6: Show and Save the Figure (Updated & More Reliable) ---

# This line automatically finds your Desktop folder on Windows, Mac, or Linux
save_folder = os.path.expanduser('~/Desktop')

# Show the plot in an interactive window
fig.show()

# Create the full path for the HTML file and save it
# MODIFIED: Updated filename
html_filename = "animated_plant_map_top20.html"
html_full_path = os.path.join(save_folder, html_filename)
fig.write_html(html_full_path)
print(f"\n💾 Interactive animation saved to: '{html_full_path}'")

# Create the full path for the JSON file and save it with error handling
# MODIFIED: Updated filename
json_filename = 'animated_map_figure_top20.json'
json_full_path = os.path.join(save_folder, json_filename)

try:
    fig.to_json(json_full_path)
    print(f"📝 Plot structure saved to: '{json_full_path}'")
except Exception as e:
    print(f"❌ FAILED to save JSON file. Error: {e}")

🌍 Full dataset loaded with 298126 records.
🔬 Analysis limited to the first 1000 records.
🔎 Converting coordinates to country names...
✅ 2-letter country code conversion complete.
🔄 Converting 2-letter codes to 3-letter codes for plotting...
✅ 3-letter code conversion complete.

Filtered dataset to focus on the top 20 plants (1000 records).

🎨 Generating Animated Choropleth world map for Plants...


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\aayus\\Desktop\\animated_plant_map_top20.html'

In [ ]:
import pandas as pd
import plotly.express as px
import reverse_geocoder as rg
import pycountry
import json

# --- Step 1: Load All Observations ---
print("📂 Loading dataset...")
df = pd.read_csv('plant_pest_dataset.csv')
print(f"🌍 Full dataset loaded with {len(df)} records.")

# ✅ Check if required columns exist
required_cols = ['latitude', 'longitude']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"❌ Missing required column: '{col}'")

# --- Step 2: Optimize Coordinate Conversion (Much Faster) ---
print("🔎 Optimizing coordinate conversion for speed...")

# Get unique lat/lon pairs to avoid re-calculating for the same location
unique_coords_df = df[['latitude', 'longitude']].drop_duplicates()
unique_coords_list = list(zip(unique_coords_df['latitude'], unique_coords_df['longitude']))

print(f"📌 Found {len(df)} total coordinates, but only {len(unique_coords_list)} are unique.")

# Perform the expensive reverse geocoding only on the unique coordinates
results = rg.search(unique_coords_list)

# Create a mapping from coordinate pair to country code
coord_to_country_map = {
    coord: res['cc'] for coord, res in zip(unique_coords_list, results)
}

# Map the country codes back to the original dataframe
df['country_code_2_letter'] = df.apply(
    lambda row: coord_to_country_map.get((row['latitude'], row['longitude'])),
    axis=1
)
print("✅ Optimized 2-letter country code conversion complete.")

# Optional: Show missing geocodes
missing_count = df['country_code_2_letter'].isna().sum()
if missing_count > 0:
    print(f"⚠️ {missing_count} records could not be reverse-geocoded.")

# --- Step 3: Convert 2-Letter to 3-Letter Codes ---
print("🔄 Converting 2-letter codes to 3-letter codes for plotting...")

def get_iso3_code(iso2_code):
    """Converts a 2-letter ISO code to a 3-letter ISO code."""
    try:
        return pycountry.countries.get(alpha_2=iso2_code).alpha_3
    except AttributeError:
        return None

df['country_code_3_letter'] = df['country_code_2_letter'].apply(get_iso3_code)
df.dropna(subset=['country_code_3_letter'], inplace=True)
print("✅ 3-letter code conversion complete.")

# Quick sanity check
print("\n🧾 Sample after geocoding:")
print(df[['latitude', 'longitude', 'country_code_2_letter', 'country_code_3_letter']].head())

# --- Step 4: Prepare Data for a Static Map ---
print("\n📊 Aggregating total sightings per country...")
total_country_counts = df.groupby('country_code_3_letter').size().reset_index(name='total_sightings')
total_country_counts.rename(columns={'country_code_3_letter': 'country'}, inplace=True)
print("✅ Aggregation complete.")

# --- Step 5: Create the Static Choropleth Map ---
print("\n🎨 Generating Static Choropleth world map...")

fig = px.choropleth(
    total_country_counts,
    locations="country",
    color="total_sightings",
    hover_name="country",
    color_continuous_scale=px.colors.sequential.Plasma,
    title="Total Global Plant Sightings"
)

fig.update_layout(
    title_x=0.5,
    geo=dict(
        showframe=False,
        showcoastlines=False,
        projection_type='natural earth'
    ),
    coloraxis_colorbar=dict(
        title="Total Sightings"
    )
)

# --- Step 6: Show and Save the Figure ---
fig.show()

# Save the HTML file
html_filename = "static_total_sightings_map.html"
fig.write_html(html_filename, include_plotlyjs='cdn')
print(f"\n💾 Static map saved to: '{html_filename}' (smaller file size)")

# ✅ Save the JSON file properly
json_filename = 'static_map_figure.json'
try:
    fig_json = fig.to_json()  # returns JSON string
    with open(json_filename, 'w', encoding='utf-8') as f:
        f.write(fig_json)
    print(f"📝 Plot structure saved to: '{json_filename}' ✅")
except Exception as e:
    print(f"❌ FAILED to save JSON file. Error: {e}")


📂 Loading dataset...
🌍 Full dataset loaded with 298126 records.
🔎 Optimizing coordinate conversion for speed...
📌 Found 298126 total coordinates, but only 288916 are unique.
✅ Optimized 2-letter country code conversion complete.
🔄 Converting 2-letter codes to 3-letter codes for plotting...
✅ 3-letter code conversion complete.

🧾 Sample after geocoding:
    latitude   longitude country_code_2_letter country_code_3_letter
0  40.799434 -111.012697                    US                   USA
1  40.690640 -110.903167                    US                   USA
2  43.939639  -87.719908                    US                   USA
3  44.741903  -65.519220                    CA                   CAN
4  42.146311  -77.131258                    US                   USA

📊 Aggregating total sightings per country...
✅ Aggregation complete.

🎨 Generating Static Choropleth world map...



💾 Static map saved to: 'static_total_sightings_map.html' (smaller file size)
📝 Plot structure saved to: 'static_map_figure.json' ✅


In [ ]:
# --- Step 4: NEW - Save the Processed Data to JSON ---
output_filename = 'phenophase_environmental_profiles.json'
df_long.to_json(output_filename, orient='records', indent=4)
print(f"\n💾 Processed data has been saved to '{output_filename}'")


💾 Processed data has been saved to 'phenophase_environmental_profiles.json'


In [15]:
import pandas as pd
import plotly.express as px

# --- Step 1: Load your dataset ---
# Make sure 'plant_pest_dataset.csv' is in the same directory as your script
try:
    df = pd.read_csv('plant_pest_dataset.csv')
    print(f"🌍 Dataset loaded with {len(df)} records.")
except FileNotFoundError:
    print("❌ Error: 'plant_pest_dataset.csv' not found. Please check the file path.")
    exit()

# --- Step 2: Prepare the data ---
# Filter out zero-values for meaningful comparison
df_filtered = df[(df['SWGDN'] > 0) & (df['PRECTOTCORR'] > 0)].copy()
print(f"🔬 Using {len(df_filtered)} records for analysis.")

# Group by phenophase and calculate the average for each environmental variable
phenophase_profile = df_filtered.groupby('phenophase')[['T2M', 'PRECTOTCORR', 'SWGDN']].mean().reset_index()

print("\n📊 Average Environmental Profile per Phenophase:")
print(phenophase_profile)

# "Melt" the data to make it easy to plot
df_long = pd.melt(
    phenophase_profile,
    id_vars='phenophase',
    var_name='Environmental Variable',
    value_name='Average Value'
)

# --- Step 3: Create the Bar Charts with a Black Theme ---
print("\n📊 Generating classic bar charts with a dark theme...")

fig = px.bar(
    df_long,
    x='phenophase',
    y='Average Value',
    color='phenophase',
    facet_col='Environmental Variable',
    text_auto='.2s',
    title="Average Environmental Conditions by Phenophase",
    template='plotly_dark'  # This line sets the theme to black
)

# --- Step 4: Customize and Show the Plot ---
fig.update_yaxes(matches=None)
fig.update_layout(title_x=0.5, showlegend=False)
fig.for_each_xaxis(lambda x: x.update(title=''))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()
print("✅ Plot generation complete.")

# --- Step 5: Save the Processed Data ---
output_filename = 'phenophase_environmental_profiles.json'
df_long.to_json(output_filename, orient='records', indent=4) # Using indent for readability
print(f"\n💾 Processed data saved to '{output_filename}'")

output_filename_gz = 'phenophase_environmental_profiles.json.gz'
df_long.to_json(output_filename_gz, orient='records', compression='gzip')
print(f"💾 Compressed JSON saved to '{output_filename_gz}'")

🌍 Dataset loaded with 298126 records.
🔬 Using 298126 records for analysis.

📊 Average Environmental Profile per Phenophase:
  phenophase         T2M  PRECTOTCORR         SWGDN
0    Budding  289.250521     0.000035   1858.931257
1  Flowering  287.487612     0.000034   9418.205752
2   Fruiting  280.113695     0.000032  22743.038304

📊 Generating classic bar charts with a dark theme...


✅ Plot generation complete.

💾 Processed data saved to 'phenophase_environmental_profiles.json'
💾 Compressed JSON saved to 'phenophase_environmental_profiles.json.gz'


In [ ]:
import pandas as pd
import plotly.express as px

# --- Step 1: Load your dataset ---
# Make sure 'plant_pest_dataset.csv' is in the same directory as your script
try:
    df = pd.read_csv('plant_pest_dataset.csv')
    print(f"🌍 Dataset loaded with {len(df)} records.")
except FileNotFoundError:
    print("❌ Error: 'plant_pest_dataset.csv' not found. Please check the file path.")
    exit()

# --- Step 2: Prepare the data ---
# Filter out zero-values for meaningful comparison
df_filtered = df[(df['SWGDN'] > 0) & (df['PRECTOTCORR'] > 0)].copy()
print(f"🔬 Using {len(df_filtered)} records for analysis.")

# Group by phenophase and calculate the average for each environmental variable
phenophase_profile = df_filtered.groupby('phenophase')[['T2M', 'PRECTOTCORR', 'SWGDN']].mean().reset_index()

print("\n📊 Average Environmental Profile per Phenophase:")
print(phenophase_profile)

# "Melt" the data to make it easy to plot
df_long = pd.melt(
    phenophase_profile,
    id_vars='phenophase',
    var_name='Environmental Variable',
    value_name='Average Value'
)

# --- Step 3: Create the Bar Charts with a Black Theme ---
print("\n📊 Generating classic bar charts with a dark theme...")

fig = px.bar(
    df_long,
    x='phenophase',
    y='Average Value',
    color='phenophase',
    facet_col='Environmental Variable',
    text_auto='.2s',
    title="Average Environmental Conditions by Phenophase",
    template='plotly_dark'  # This line sets the theme to black
)

# --- Step 4: Customize and Show the Plot ---
fig.update_yaxes(matches=None)
fig.update_layout(title_x=0.5, showlegend=False)
fig.for_each_xaxis(lambda x: x.update(title=''))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()
print("✅ Plot generation complete.")



🌍 Dataset loaded with 298126 records.
🔬 Using 298126 records for analysis.

📊 Average Environmental Profile per Phenophase:
  phenophase         T2M  PRECTOTCORR         SWGDN
0    Budding  289.250521     0.000035   1858.931257
1  Flowering  287.487612     0.000034   9418.205752
2   Fruiting  280.113695     0.000032  22743.038304

📊 Generating classic bar charts with a dark theme...


✅ Plot generation complete.

💾 Processed data saved to 'phenophase_environmental_profiles.json'
💾 Compressed JSON saved to 'phenophase_environmental_profiles.json.gz'


In [17]:
import pandas as pd
import plotly.express as px

# --- Step 1: Load your dataset ---
try:
    df = pd.read_csv('plant_pest_dataset.csv')
    print(f"🌍 Dataset loaded with {len(df)} records.")
except FileNotFoundError:
    print("❌ Error: 'plant_pest_dataset.csv' not found. Please check the file path.")
    exit()

# --- Step 2: Identify the Top 10 Pests ---
# This keeps the visualization clean and focused on the most significant pests.
top_10_pests = df['scientific_name_pest'].value_counts().nlargest(10).index
df_filtered = df[df['scientific_name_pest'].isin(top_10_pests)]
print(f"🔬 Focusing on the 10 most common pests for clarity.")


# --- Step 3: Create an Environmental Profile for Each Pest ---
# Group by each pest and find the average conditions when it was sighted.
# We also count the number of sightings for each pest.
pest_profile = df_filtered.groupby('scientific_name_pest').agg(
    avg_temp=('T2M', 'mean'),
    avg_precip=('PRECTOTCORR', 'mean'),
    avg_sunlight=('SWGDN', 'mean'),
    sighting_count=('scientific_name_pest', 'size')
).reset_index()

print("\n📊 Environmental Profile for Top 10 Pests:")
print(pest_profile)


# --- Step 4: Create the Scatter Plot ---
print("\n📈 Generating scatter plot to show pest environmental niches...")

fig = px.scatter(
    pest_profile,
    x='avg_temp',
    y='avg_precip',
    size='sighting_count',         # Bubble size reflects sighting frequency
    color='scientific_name_pest',  # Each pest gets a unique color
    hover_name='scientific_name_pest', # Show pest name on hover
    size_max=60,                   # Control the maximum bubble size
    template='plotly_dark',        # Use the requested black theme
    title="Environmental Niche of Top 10 Pests",
    labels={
        "avg_temp": "Average Temperature (°C) During Sightings",
        "avg_precip": "Average Precipitation (mm/day) During Sightings",
        "scientific_name_pest": "Pest Species"
    }
)

# --- Step 5: Customize and Show the Plot ---
fig.update_layout(
    title_x=0.5,
    legend_title="Pest Species"
)
fig.show()
print("✅ Plot generation complete.")

🌍 Dataset loaded with 298126 records.
🔬 Focusing on the 10 most common pests for clarity.

📊 Environmental Profile for Top 10 Pests:
        scientific_name_pest    avg_temp  avg_precip  avg_sunlight  \
0        Agrilus planipennis  286.895464    0.000036   1455.177727   
1      Brevicoryne brassicae  287.940783    0.000025    401.537078   
2            Cydia pomonella  289.849983    0.000028   1367.228554   
3  Daktulosphaira vitifoliae  293.038476    0.000032    885.171801   
4         Drosophila suzukii  286.093724    0.000032    439.311075   
5             Empoasca fabae  293.426634    0.000035    446.192772   
6            Icerya purchasi  288.469855    0.000026   1675.411539   
7  Leptinotarsa decemlineata  291.425773    0.000033   1795.262632   
8         Ostrinia nubilalis  291.390286    0.000036    654.569431   
9      Plodia interpunctella  287.883036    0.000034   2640.730238   

   sighting_count  
0           39380  
1           20091  
2           12626  
3           2371

✅ Plot generation complete.


🌍 Dataset loaded with 298126 records.
🔬 Focusing analysis on the 10 most frequently sighted pests.

📊 Average Environmental Profile for Top 10 Pests:
        scientific_name_pest  avg_temp  avg_precip  avg_sunlight  \
0        Agrilus planipennis    286.90         0.0       1455.18   
1      Brevicoryne brassicae    287.94         0.0        401.54   
2            Cydia pomonella    289.85         0.0       1367.23   
3  Daktulosphaira vitifoliae    293.04         0.0        885.17   
4         Drosophila suzukii    286.09         0.0        439.31   
5             Empoasca fabae    293.43         0.0        446.19   
6            Icerya purchasi    288.47         0.0       1675.41   
7  Leptinotarsa decemlineata    291.43         0.0       1795.26   
8         Ostrinia nubilalis    291.39         0.0        654.57   
9      Plodia interpunctella    287.88         0.0       2640.73   

   sighting_count  
0           39380  
1           20091  
2           12626  
3           23710  
4

✅ Plot generation complete.


In [19]:
import pandas as pd
import plotly.express as px

# --- Step 1: Load your dataset ---
# This script assumes 'plant_pest_dataset.csv' is in the same directory.
try:
    df = pd.read_csv('plant_pest_dataset.csv')
    print(f"🌍 Dataset loaded with {len(df)} records.")
except FileNotFoundError:
    print("❌ Error: 'plant_pest_dataset.csv' not found. Please check the file path.")
    exit()

# --- Step 2: Identify the Top 10 Pests for Analysis ---
# We focus on the most common pests to create a clear and readable chart.
top_10_pests = df['scientific_name_pest'].value_counts().nlargest(10).index
df_filtered = df[df['scientific_name_pest'].isin(top_10_pests)].copy()
print(f"🔬 Focusing analysis on the 10 most frequently sighted pests.")


# --- Step 3: Calculate the Environmental Profile for Each Pest ---
# For each pest, we group all its sightings and calculate the average
# environmental conditions and the total number of sightings.
pest_profile = df_filtered.groupby('scientific_name_pest').agg(
    avg_temp=('T2M', 'mean'),
    avg_precip=('PRECTOTCORR', 'mean'),
    avg_sunlight=('SWGDN', 'mean'),
    sighting_count=('scientific_name_pest', 'size')
).reset_index()

print("\n📊 Average Environmental Profile for Top 10 Pests:")
print(pest_profile.round(2)) # Displaying with rounded numbers for clarity


# --- Step 4: Create the Scatter Plot to Visualize Pest Niches ---
print("\n📈 Generating scatter plot to show pest environmental preferences...")

fig = px.scatter(
    pest_profile,
    x='avg_temp',
    y='avg_precip',
    size='sighting_count',         # Bubble size shows how common the pest is
    color='scientific_name_pest',  # Each pest gets a unique color
    hover_name='scientific_name_pest', # Shows pest name when you mouse over
    size_max=60,                   # Controls the size of the largest bubble
    template='plotly_dark',        # Sets the black theme
    title="Environmental Niche: Preferred Conditions of Top 10 Pests",
    labels={
        "avg_temp": "Average Temperature (°C) During Sightings",
        "avg_precip": "Average Precipitation (mm/day) During Sightings",
        "scientific_name_pest": "Pest Species",
        "sighting_count": "Total Sightings"
    }
)

# --- Step 5: Customize and Show the Plot ---
fig.update_layout(
    title_x=0.5, # Center the title
    legend_title="Pest Species"
)

fig.show()
print("✅ Plot generation complete.")

🌍 Dataset loaded with 298126 records.
🔬 Focusing analysis on the 10 most frequently sighted pests.

📊 Average Environmental Profile for Top 10 Pests:
        scientific_name_pest  avg_temp  avg_precip  avg_sunlight  \
0        Agrilus planipennis    286.90         0.0       1455.18   
1      Brevicoryne brassicae    287.94         0.0        401.54   
2            Cydia pomonella    289.85         0.0       1367.23   
3  Daktulosphaira vitifoliae    293.04         0.0        885.17   
4         Drosophila suzukii    286.09         0.0        439.31   
5             Empoasca fabae    293.43         0.0        446.19   
6            Icerya purchasi    288.47         0.0       1675.41   
7  Leptinotarsa decemlineata    291.43         0.0       1795.26   
8         Ostrinia nubilalis    291.39         0.0        654.57   
9      Plodia interpunctella    287.88         0.0       2640.73   

   sighting_count  
0           39380  
1           20091  
2           12626  
3           23710  
4

✅ Plot generation complete.


In [22]:
!pip install statsmodels

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.6 MB 2.1 MB/s eta 0:00:05
   -------- ------------------------------- 2.1/9.6 MB 3.9 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.6 MB 5.1 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/9.6 MB 5.8 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.6 MB 6.1 MB/s eta 0:00:01
   ------------------------------------- -- 8.9/9.6 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 6.4 MB/s  0:00:01

   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ------------------------------------

In [28]:
!pip install kaleido


   ---------------------------------------- 0/9 [simplejson]
   ---------------------------------------- 0/9 [simplejson]
   ---------------------------------------- 0/9 [simplejson]
   ---------------------------------------- 0/9 [simplejson]
   ---------------------------------------- 0/9 [simplejson]
   ---------------------------------------- 0/9 [simplejson]
   ---- ----------------------------------- 1/9 [pluggy]
   ---- ----------------------------------- 1/9 [pluggy]
   ----------------- ---------------------- 4/9 [iniconfig]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   ---------------------- ----------------- 5/9 [pytest]
   

In [25]:
import pandas as pd

# Load your dataset
df = pd.read_csv('plant_pest_dataset.csv')

# --- Re-create the climate zone function ---
def get_climate_zone(latitude):
    lat = abs(latitude)
    if lat <= 23.5:
        return 'Tropical'
    elif 23.5 < lat <= 66.5:
        return 'Temperate'
    else:
        return 'Arctic/Polar'

# Create the 'climate_zone' column
df['climate_zone'] = df['latitude'].apply(get_climate_zone)

# --- This is the important part ---
# Filter the DataFrame to show ONLY the rows that are NOT 'Temperate'
non_temperate_data = df[df['climate_zone'] != 'Temperate']

print(f"Found {len(non_temperate_data)} rows that are either Tropical or Arctic/Polar.")
print("\nHere is a sample of those rows:")

# Display the first 10 rows found
display(non_temperate_data.head(10))

Found 959 rows that are either Tropical or Arctic/Polar.

Here is a sample of those rows:


,id,observed_on,latitude,longitude,phenophase,scientific_name,common_name,taxon_id,doy,biome,T2M,PRECTOTCORR,SWGDN,biome_cat,scientific_name_pest,common_name_pest,climate_zone
72,306548660,2025-08-15,66.939370,-53.660745,Flowering,Achillea millefolium,common yarrow,52821,227,29.0,278.09120,0.000037,133.426860,H,Plodia interpunctella,Indian-Meal Moth,Arctic/Polar
85,311620767,2025-08-13,66.939383,-53.660600,Flowering,Achillea millefolium,common yarrow,52821,225,29.0,278.09120,0.000037,133.426860,H,Plodia interpunctella,Indian-Meal Moth,Arctic/Polar
205,301927181,2025-07-28,69.249817,-53.527608,Flowering,Achillea millefolium,common yarrow,52821,209,29.0,278.02805,0.000034,134.183700,H,Diabrotica virgifera,Western Corn Rootworm,Arctic/Polar
385,296836287,2025-07-10,66.606950,19.822921,Flowering,Achillea millefolium,common yarrow,52821,191,27.0,290.84143,0.000020,249.872250,H,Popillia japonica,Japanese Beetle,Arctic/Polar
834,270492706,2024-12-31,9.559033,-83.724900,Flowering,Achillea millefolium,common yarrow,52821,366,29.0,294.10600,0.000039,218.810970,H,Diabrotica virgifera,Western Corn Rootworm,Tropical
1140,238874520,2024-08-30,78.195173,15.560806,Flowering,Achillea millefolium,common yarrow,52821,243,29.0,273.68307,0.000032,32.329758,H,Ceratitis capitata,Mediterranean Fruit Fly,Arctic/Polar
1225,235873217,2024-08-15,70.432059,24.506546,Flowering,Achillea millefolium,common yarrow,52821,228,27.0,287.58057,0.000023,172.000050,H,Daktulosphaira vitifoliae,Grape Phylloxera,Arctic/Polar
1234,238094077,2024-08-14,69.166840,35.143242,Flowering,Achillea millefolium,common yarrow,52821,227,27.0,286.93036,0.000012,186.290480,H,Plutella xylostella,Diamondback Moth,Arctic/Polar
1326,233728614,2024-08-04,66.543521,25.849513,Flowering,Achillea millefolium,common yarrow,52821,217,27.0,288.66684,0.000037,170.625690,H,Lymantria dispar,Spongy Moth,Arctic/Polar
1371,234323615,2024-08-01,69.023056,32.946944,Flowering,Achillea millefolium,common yarrow,52821,214,27.0,287.57996,0.000016,184.237050,H,Drosophila suzukii,Spotted-winged Drosophila,Arctic/Polar


In [29]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Step 1: Load and Prepare Data ---
df = pd.read_csv('plant_pest_dataset.csv')
print(f"🌍 Full dataset loaded with {len(df)} records.")

# Convert 'observed_on' to datetime and drop rows with invalid dates
df['observed_on'] = pd.to_datetime(df['observed_on'], errors='coerce')
df.dropna(subset=['observed_on', 'doy', 'T2M'], inplace=True)

# Create 'year' column
df['year'] = df['observed_on'].dt.year

# --- Step 2: Filter for the Last 10 Years ---
# Using a fixed year for reproducibility
start_year = 2015

df_recent = df[df['year'] >= start_year].copy()
print(f"🗓️ Filtered to {len(df_recent)} records from {start_year} onwards.")

# --- Step 3: Aggregate Data by Year ---
# Calculate the average Day of Year and average Temperature for each year
yearly_trends = df_recent.groupby('year').agg(
    avg_doy=('doy', 'mean'),
    avg_temp=('T2M', 'mean')
).reset_index()

print("\n📊 Aggregated yearly averages for phenology and temperature:")
print(yearly_trends)


# --- Step 4: Create the Dual-Axis Combination Chart ---
print("\n🎨 Generating the final trend comparison plot...")

# Create a figure with a secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# --- Add Phenophase Trend (Bar Chart) ---
fig.add_trace(
    go.Bar(
        x=yearly_trends['year'],
        y=yearly_trends['avg_doy'],
        name='Phenophase Timing (Avg. Day of Year)',
        marker_color='skyblue'
    ),
    secondary_y=False,
)

# --- Add Temperature Trend (Line Chart) ---
fig.add_trace(
    go.Scatter(
        x=yearly_trends['year'],
        y=yearly_trends['avg_temp'],
        name='Avg. Temperature',
        mode='lines+markers',
        line=dict(color='crimson', width=3)
    ),
    secondary_y=True,
)

# --- Step 5: Improve and Style the Layout ---
fig.update_layout(
    template='plotly_dark',
    title_text='<b>Phenophase Timing vs. Temperature Shift Over the Last 10 Years</b>',
    title_x=0.5,
    xaxis_title='Year',
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right", x=1
    )
)

# Set y-axes titles
fig.update_yaxes(
    title_text='<b>Avg. Day of Year</b> (Lower is Earlier)',
    secondary_y=False,
    color='skyblue'
)
fig.update_yaxes(
    title_text='<b>Avg. Temperature (°C)</b>',
    secondary_y=True,
    color='crimson'
)

# --- Step 6: Show and Save the Figure ---
fig.show()

# ADD THIS LINE TO SAVE THE FIGURE AS A PNG FILE
fig.write_image("phenology_vs_temp_shift.png")
print("✅ Chart saved as phenology_vs_temp_shift.png")

🌍 Full dataset loaded with 298126 records.
🗓️ Filtered to 298126 records from 2015 onwards.

📊 Aggregated yearly averages for phenology and temperature:
    year     avg_doy    avg_temp
0   2015  331.250000  284.075050
1   2016  144.603448  297.791834
2   2017  160.583333  292.990981
3   2018  167.061453  288.760508
4   2019  158.724238  290.300380
5   2020  220.150551  286.568038
6   2021  181.454231  289.217539
7   2022  195.043536  289.794240
8   2023  182.660086  290.399226
9   2024  190.571157  290.651892
10  2025  155.004472  274.750487

🎨 Generating the final trend comparison plot...


ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


In [44]:
!pip install --upgrade kaleido

In [31]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Step 1: Load and Prepare Data ---
df = pd.read_csv('plant_pest_dataset.csv')
print(f"🌍 Full dataset loaded with {len(df)} records.")

# Convert 'observed_on' to datetime and drop rows with invalid dates
df['observed_on'] = pd.to_datetime(df['observed_on'], errors='coerce')
df.dropna(subset=['observed_on', 'doy', 'T2M'], inplace=True)

# Create 'year' column
df['year'] = df['observed_on'].dt.year

# --- Step 2: Filter for the Last 10 Years ---
# Using a fixed year for reproducibility
start_year = 2015

df_recent = df[df['year'] >= start_year].copy()
print(f"🗓️ Filtered to {len(df_recent)} records from {start_year} onwards.")

# --- Step 3: Aggregate Data by Year ---
# Calculate the average Day of Year and average Temperature for each year
yearly_trends = df_recent.groupby('year').agg(
    avg_doy=('doy', 'mean'),
    avg_temp=('T2M', 'mean')
).reset_index()

print("\n📊 Aggregated yearly averages for phenology and temperature:")
print(yearly_trends)


# --- Step 4: Create the Dual-Axis Combination Chart ---
print("\n🎨 Generating the final trend comparison plot...")

# Create a figure with a secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# --- Add Phenophase Trend (Bar Chart) ---
fig.add_trace(
    go.Bar(
        x=yearly_trends['year'],
        y=yearly_trends['avg_doy'],
        name='Phenophase Timing (Avg. Day of Year)',
        marker_color='skyblue'
    ),
    secondary_y=False,
)

# --- Add Temperature Trend (Line Chart) ---
fig.add_trace(
    go.Scatter(
        x=yearly_trends['year'],
        y=yearly_trends['avg_temp'],
        name='Avg. Temperature',
        mode='lines+markers',
        line=dict(color='crimson', width=3)
    ),
    secondary_y=True,
)

# --- Step 5: Improve and Style the Layout ---
fig.update_layout(
    template='plotly_dark',
    title_text='<b>Phenophase Timing vs. Temperature Shift Over the Last 10 Years</b>',
    title_x=0.5,
    xaxis_title='Year',
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right", x=1
    )
)

# Set y-axes titles
fig.update_yaxes(
    title_text='<b>Avg. Day of Year</b> (Lower is Earlier)',
    secondary_y=False,
    color='skyblue'
)
fig.update_yaxes(
    title_text='<b>Avg. Temperature (°C)</b>',
    secondary_y=True,
    color='crimson'
)

# --- Step 6: Show and Save the Figure ---
fig.show()

# This line saves the figure as a PNG file
fig.write_image("phenology_vs_temp_shift.png")
print("✅ Chart saved as phenology_vs_temp_shift.png")

🌍 Full dataset loaded with 298126 records.
🗓️ Filtered to 298126 records from 2015 onwards.

📊 Aggregated yearly averages for phenology and temperature:
    year     avg_doy    avg_temp
0   2015  331.250000  284.075050
1   2016  144.603448  297.791834
2   2017  160.583333  292.990981
3   2018  167.061453  288.760508
4   2019  158.724238  290.300380
5   2020  220.150551  286.568038
6   2021  181.454231  289.217539
7   2022  195.043536  289.794240
8   2023  182.660086  290.399226
9   2024  190.571157  290.651892
10  2025  155.004472  274.750487

🎨 Generating the final trend comparison plot...


ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


In [39]:
import pandas as pd
import plotly.express as px

# --- Load and Prepare Data ---
df = pd.read_csv('plant_pest_dataset.csv')

# --- ADD THIS LINE TO SEE YOUR REAL COLUMN NAMES ---
print("Your actual column names are:")
print(df.columns)
# --- LOOK AT THE OUTPUT OF THE LINE ABOVE ---

# Now, find the correct name for precipitation in the list and use it below.
# I will assume the correct name is 'PRECTOT' for this example.
precipitation_column_name = 'PRECTOTCORR' # <-- CHANGE THIS IF IT'S DIFFERENT

df['observed_on'] = pd.to_datetime(df['observed_on'], errors='coerce')
# Use the variable here to avoid errors
df.dropna(subset=['observed_on', precipitation_column_name], inplace=True)

# --- Aggregate Data by Day ---
daily_precipitation = df.groupby(df['observed_on'].dt.date)[precipitation_column_name].sum().reset_index()

# --- Create the Bar Chart ---
fig = px.bar(
    daily_precipitation,
    x='observed_on',
    y=precipitation_column_name,
    title='Total Daily Precipitation',
    labels={'observed_on': 'Date', precipitation_column_name: 'Total Precipitation (mm)'}
)

fig.update_layout(template='plotly_dark', title_x=0.5)
fig.show()

Your actual column names are:
Index(['id', 'observed_on', 'latitude', 'longitude', 'phenophase',
       'scientific_name', 'common_name', 'taxon_id', 'doy', 'biome', 'T2M',
       'PRECTOTCORR', 'SWGDN', 'biome_cat', 'scientific_name_pest',
       'common_name_pest'],
      dtype='object')


In [33]:
import pandas as pd
import plotly.express as px

# --- Load and Prepare Data ---
df = pd.read_csv('plant_pest_dataset.csv')
df['observed_on'] = pd.to_datetime(df['observed_on'], errors='coerce')
df.dropna(subset=['observed_on', 'phenophase'], inplace=True)

# --- Aggregate Data ---
# We group by both day and phenophase, then count the occurrences
phenophase_counts = df.groupby([df['observed_on'].dt.date, 'phenophase']).size().reset_index(name='count')

# --- Create the Stacked Bar Chart ---
fig = px.bar(
    phenophase_counts,
    x='observed_on',
    y='count',
    color='phenophase', # This creates the stacking
    title='Daily Observed Plant Phenophases',
    labels={'observed_on': 'Date', 'count': 'Number of Observations'}
)

fig.update_layout(template='plotly_dark', title_x=0.5, barmode='stack')
fig.show()

In [34]:
import pandas as pd
import plotly.graph_objects as go

# --- Load and Prepare Data ---
df = pd.read_csv('plant_pest_dataset.csv')
df['observed_on'] = pd.to_datetime(df['observed_on'], errors='coerce')
df.dropna(subset=['observed_on', 'T2M'], inplace=True)

# --- Aggregate and Calculate Rolling Stats ---
# Get the average temperature for each day
daily_temp = df.groupby(df['observed_on'].dt.date)['T2M'].mean().reset_index()

# Calculate a 14-day rolling average to smooth the trend
window = 14
daily_temp['T2M_rolling_avg'] = daily_temp['T2M'].rolling(window=window).mean()
# Calculate the rolling standard deviation to define the "normal" range
daily_temp['T2M_rolling_std'] = daily_temp['T2M'].rolling(window=window).std()

# Define the upper and lower bounds of the normal range
daily_temp['upper_bound'] = daily_temp['T2M_rolling_avg'] + daily_temp['T2M_rolling_std']
daily_temp['lower_bound'] = daily_temp['T2M_rolling_avg'] - daily_temp['T2M_rolling_std']


# --- Create the Chart with Shaded Area ---
fig = go.Figure()

# Add the upper and lower bound traces first for the shaded area
fig.add_trace(go.Scatter(
    x=daily_temp['observed_on'], y=daily_temp['upper_bound'],
    mode='lines', line=dict(width=0), showlegend=False
))
fig.add_trace(go.Scatter(
    x=daily_temp['observed_on'], y=daily_temp['lower_bound'],
    mode='lines', line=dict(width=0), showlegend=False,
    fill='tonexty', fillcolor='rgba(255, 255, 255, 0.1)', # This creates the shaded area
    name='Normal Range'
))

# Add the main temperature trend line
fig.add_trace(go.Scatter(
    x=daily_temp['observed_on'], y=daily_temp['T2M_rolling_avg'],
    mode='lines', line=dict(color='cyan', width=3),
    name='14-Day Avg. Temperature'
))

fig.update_layout(
    title='Daily Temperature Trend with Normal Range',
    xaxis_title='Date', yaxis_title='Temperature (°C)',
    template='plotly_dark', title_x=0.5
)
fig.show()

In [46]:
import pandas as pd
import plotly.express as px

# --- Step 1: Load and Prepare Data ---
print("📂 Loading dataset...")
df = pd.read_csv('plant_pest_dataset.csv')
print(f"🌍 Full dataset loaded with {len(df)} records.")

# --- Step 2: Select a High-Risk Pest to Map ---
pest_to_map = 'Leptinotarsa decemlineata'
df_pest = df[df['scientific_name_pest'] == pest_to_map].copy()
print(f"🔬 Focusing on {len(df_pest)} records for the pest: {pest_to_map}")

# Ensure we have the necessary location data
df_pest.dropna(subset=['latitude', 'longitude'], inplace=True)

# --- Step 3: Create the Density Risk Map ---
print("\n🎨 Generating the Pest Density Risk Map...")

# CORRECTED: Using density_map with the 'mapbox_style' parameter
fig = px.density_mapbox(
    df_pest,
    lat='latitude',
    lon='longitude',
    radius=10,
    center=dict(lat=40, lon=-95),
    zoom=3,
    mapbox_style="stamen-terrain", # This is the correct parameter name
    color_continuous_scale="Reds",
    title=f"<b>Risk Map: High Density Areas for {pest_to_map}</b>"
)

# --- Step 4: Customize and Show the Map ---
fig.update_layout(
    title_x=0.5,
    template='plotly_dark',
    coloraxis_colorbar=dict(
        title="Sighting<br>Density"
    )
)

fig.show()



📂 Loading dataset...
🌍 Full dataset loaded with 298126 records.
🔬 Focusing on 15920 records for the pest: Leptinotarsa decemlineata

🎨 Generating the Pest Density Risk Map...


C:\Users\aayus\AppData\Local\Temp\ipykernel_4692\2835364665.py:21: DeprecationWarning:

*density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

